# GPU benchmark on the verified NZ snapshot

This notebook does exactly one thing: **measure** training time/epoch, validation time, and peak GPU memory for a short, properly batched run on the real, verified snapshot -- not estimate, measure. Nothing here commits to a full training run; that decision comes after these numbers exist, once the data has been confirmed to load correctly here too.

Prerequisite (done on the local machine, not here): `data/nz_pilot_snapshot` was built by `scripts/colab/build_snapshot.py` -- every tile independently checksum- and schema-verified, cross-split geographic separation and parcel-id overlap both confirmed clean. Zip it and upload to Drive before running this:

```bash
cd /path/to/SIH_2026
zip -r nz_pilot_snapshot.zip data/nz_pilot_snapshot
```
Then upload `nz_pilot_snapshot.zip` to `My Drive/` (the Drive app or web uploader is far faster than uploading through the Colab file browser for a multi-GB file).

In [ ]:
!git clone https://github.com/SatvikSaluja/SIH_2026.git
%cd SIH_2026
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

In [ ]:
# Verify CUDA is ACTUALLY available before anything else -- a benchmark
# run silently on CPU because the runtime type wasn't set to GPU would
# produce numbers that look like a GPU result but aren't.
import torch
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > select a GPU, then rerun"
print("GPU:", torch.cuda.get_device_name(0))
print("Total memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/nz_pilot_snapshot.zip -d /content/
!ls /content/nz_pilot_snapshot | wc -l  # should be ~710 (708 train + val + test + manifest.json)

In [ ]:
# Confirm the data actually loads correctly HERE before trusting any
# timing number from it -- a manifest/schema mismatch that somehow
# didn't surface locally must not surface for the first time inside a
# timed run.
import sys
sys.path.insert(0, '/content/SIH_2026/scripts/colab')
from train_real import Patches
train = Patches('/content/nz_pilot_snapshot', 'train')
val = Patches('/content/nz_pilot_snapshot', 'val')
print(f"{len(train)} training patches, {len(val)} validation patches -- loaded without error")

In [ ]:
# The actual measurement. batch_size=32 is a starting point, not tuned --
# the point of this cell is the numbers it prints, not this choice.
!python /content/SIH_2026/scripts/colab/benchmark_gpu.py \
  --data /content/nz_pilot_snapshot \
  --epochs 3 \
  --batch-size 32

## Reading the result

`benchmark_result.json` (also printed above) has `mean_train_seconds_per_epoch`, `mean_val_seconds`, and `peak_gpu_memory_mb`, measured, not guessed. To project a full run's wall-clock time honestly:

```
estimated_full_run = (target_epochs / 3) * mean_train_seconds_per_epoch
```

using the SAME 708-tile snapshot -- this notebook does not extrapolate to a larger tile count, because that would be a second, different assumption stacked on top of a measured one. If more tiles are added later, rerun this notebook against the new snapshot rather than scaling this number.

**Do not treat a clean run here as license to start a long unattended job.** Check `peak_gpu_memory_mb` against the GPU's total memory (cell 2) with headroom for a larger batch size if one gets used later, and look at whether `val_loss` moved sensibly across the 3 epochs -- a NaN or a value that exploded means something is still wrong with the data or setup, not something a longer run will fix.

## Real training run (2141-tile expanded snapshot)

The benchmark above confirmed the pipeline works end-to-end and gave real per-epoch timing on the 708-tile snapshot. This section runs the actual training job on the expanded, independently re-verified snapshot -- 2127 tiles kept (2125 train / 1 val / 1 test) out of 2141 candidates, same reserved val/test tiles as the benchmark snapshot so results stay comparable. Confirmed locally by actually loading it: 75612 / 26 / 32 patches on train/val/test.

If this is a fresh runtime, run cells 1-2 above first (clone + CUDA check) -- you don't need to run the benchmark itself (cells 3-6) first.

Prerequisite (done on the local machine): `data/nz_pilot_snapshot_v3` was rebuilt by `build_snapshot.py` after a real bug fix -- a stale-code carryover from a since-fixed version of `expand_full.py` had dropped val/test entirely; caught before this ever reached Colab. Upload `nz_pilot_snapshot_v3.zip` AND the `last.pt.best` checkpoint (the synthetic-pretrained warm-start source) to `My Drive/` before running this.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/nz_pilot_snapshot_v3.zip -d /content/
!ls /content/nz_pilot_snapshot_v3 | wc -l  # should be ~2128 (2125 train + val + test + manifest.json)

In [ ]:
# Same check as cell 4, but on all three splits -- val and test only
# just started existing in this snapshot (the bug above had dropped both),
# so confirming they actually load here matters more than usual.
import sys
sys.path.insert(0, '/content/SIH_2026/scripts/colab')
from train_real import Patches
for split in ('train', 'val', 'test'):
    n = len(Patches('/content/nz_pilot_snapshot_v3', split))
    print(f'{split}: {n} patches')

In [ ]:
# Fresh warm-start from the ORIGINAL synthetic-pretrained checkpoint, not
# from nz_real_run1's result -- that run was already overfitting (train
# loss went negative while val loss got worse from ~epoch 15 on), so
# starting the bigger run from it would carry that tendency forward.
# 30 epochs, not 50: the first real run's useful signal was exhausted by
# ~epoch 20-25; this checks whether 3x more data pushes that further out
# before spending the full budget on it.
!python /content/SIH_2026/scripts/colab/train_real.py \
  --data /content/nz_pilot_snapshot_v3 \
  --out /content/drive/MyDrive/nz_real_run2 \
  --epochs 30 \
  --batch-size 32 \
  --warm-start /content/drive/MyDrive/last.pt.best

## What to watch for while it runs

Each epoch prints `precision`/`recall`/`f1` for that epoch's validation as it happens -- no need to wait for the end. Three honest scenarios:

- **F1 starts moving above 0 within ~epoch 15-20** -- the data-scale fix is working; let it run the full 30.
- **F1 still flat at 0.0 by epoch 25**, matching where the 708-tile run already plateaued, even with 3x more data -- a real signal that something structural needs fixing, not more data.
- **val_loss trending down steadily even with F1 still 0** -- partial progress, worth extending via `--resume` before concluding anything.

**Time estimate, not measured:** ~2.7x more patches than the benchmark's 708-tile run -> roughly 2.7x its measured ~195s/epoch, so ~9 min/epoch, ~4.5 hours for 30 epochs. Confirm against the first couple of *actual* printed epoch times before trusting this projection -- same rule as the benchmark itself. `--resume` picks back up cleanly if a free-tier session disconnects first.